In [1]:
import numpy as np
import h5py

import hpcrec

# Read HOSM results

Read the wave elevation and velocity potential at the free surface from HOSM (DNV WAMOD)

In [7]:
with h5py.File("data/hosm_output.h5", "r") as f:
    for key in ["dkx", "dx", "h", "ndim", "nx", "xdom"]:
        val = f.attrs[key].item()
        print(f"{key}: {val}")

    dk = f.attrs["dkx"].item()
    L = f.attrs["xdom"].item()
    nx = f.attrs["nx"].item()
    depth = f.attrs["h"].item()

    # Equally spaced x-grid points (peridic domain, endpoint=startpoint)
    xgrid = np.linspace(0, L, nx, endpoint=False)

    # Wave elevation at all x-grid points from WAMOD
    eta_all = f["spatial_fields"]["eta"][:]

    # Velocity potential at all x-grid points (at z=eta) from WAMOD
    psi_all = f["spatial_fields"]["psi"][:]

dkx: 0.00043981148016716576
dx: 13.951257364201572
h: 100.0
ndim: 1
nx: 1024
xdom: 14286.08754094241


In [9]:
print(eta_all.shape, psi_all.shape)
time_index = 0
eta = eta_all[time_index, :].squeeze()
psi = psi_all[time_index, :].squeeze()
print(eta.shape, psi.shape)

(2, 1024, 1) (2, 1024, 1)
(1024,) (1024,)


# Reconstruct the wave kinematics using HPC

Create a regular domain (top boundary follows the wave, lower layers blend towards the flat bottom).
The grid resolution is given by the HOSM solution in the x-direction and we select to have Nz points
in the z-direction (computing Nz by aiming for dx=dy at the free surface, if the free surface was
exactly at z=0). The bottom is always at z=-depth.

Problem statement

* Find phi, the velocity potential in the fluid domain given eta, the free surface elevation, and
  psi, the velocity potential at the free surface.
* We know the position of all boundaries (z=-depth, z=eta, x=0, x=L) so the geometry in known.
* The solution is periodic in space (x-direction), so eta(x=0) == eta(x=L) and the same for the
  velocity potential, psi
* We know the boundary condition. Top BC is Dirichlet phi=psi, left and right are periodic, bottom
  is the normal bottom boundary condition

When we have found $\phi(x,z)$ then we can find the velocity everywhere in the fluid as $\nabla \phi$